In [1]:
!pip install tiktoken > \dev\null

In [2]:
import re
import torch
from torch.utils.data import Dataset, DataLoader
import tiktoken
import urllib.request
from importlib.metadata import version

tokenizer = tiktoken.get_encoding('gpt2')

In [3]:
#  DOWNLOAD THE DATASET AND SAVE IT TO "the-verdict.txt"
def download_data(url: str = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"):
  file_path = "the-verdict.txt"
  urllib.request.urlretrieve(url, file_path)
  with open('the-verdict.txt', 'r', encoding='utf-8') as f:
    return f.read()

raw_text = download_data()

#  PREPROCESSING
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]

#  DEFINE VOCABULARY
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)  # 1130
vocab = {token:integer for integer, token in  enumerate(all_words)}
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
vocab = {token:integer for integer, token in enumerate(all_tokens)}

#  ENCODING TEXT WITH BPE tokenizer
enc_text = tokenizer.encode(raw_text)
len(enc_text)  # 5145

5145

In [4]:
class SimpleTokenizer:
  def __init__(self, vocab: dict) -> None:
    self.str_to_int = vocab
    self.int_to_str = {i:s for s, i in vocab.items()}

  def encode(self, text: str):
    preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
    preprocessed = [item.strip() for item in preprocessed if item.strip()]

    preprocessed = [item if item in self.str_to_int
                    else "<|unk|>" for item in preprocessed]

    ids = [self.str_to_int[s] for s in preprocessed]

    return ids

  def decode(self, ids: list[int]):
    text = " ".join([self.int_to_str[i] for i in ids])
    text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
    return text

#  tokenizer = SimpleTokenizer(vocab)
#  print(tokenizer.decode(tokenizer.encode("I like computer.<|endoftext|>")))

In [5]:
class GPTDatasetV1(Dataset):
  def __init__(self, txt: str, tokenizer: object, max_lenght: int, stride: int):
    self.input_ids = []
    self.target_ids = []

    token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

    for i in range(0, len(token_ids) - max_lenght, stride):
      input_chunk = token_ids[i:i+max_lenght]
      target_chunk = token_ids[i+1:i+max_lenght+1]
      self.input_ids.append(torch.tensor(input_chunk))
      self.target_ids.append(torch.tensor(target_chunk))

  def __len__(self):
    return len(self.input_ids)

  def __getitem__(self, idx):
    return self.input_ids[idx], self.target_ids[idx]

In [6]:
def create_dataloader_v1(txt, batch_size=4, max_lenght=256, stride=128, shuffle=True, drop_last=True, num_workers=0):
  tokenizer = tiktoken.get_encoding('gpt2')
  dataset = GPTDatasetV1(txt, tokenizer, max_lenght, stride)
  dataloader = DataLoader(
      dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers
  )
  return dataloader

In [7]:
max_lenght = 4
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_lenght=max_lenght, stride=4, shuffle=False)

In [8]:
vocab_size = 50257
output_dim = 256

data_iter = iter(dataloader)
inputs, targets = next(data_iter)

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
token_embeddings = token_embedding_layer(inputs)

context_lenght = max_lenght
pos_embedding_layer = torch.nn.Embedding(context_lenght, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(context_lenght))

input_embeddings = token_embeddings + pos_embeddings

print(input_embeddings.shape)

torch.Size([8, 4, 256])


In [30]:
import torch

inputs = torch.tensor(
    [[0.43, 0.15, 0.89],
     [0.55, 0.87, 0.66],
     [0.57, 0.85, 0.64],
     [0.22, 0.58, 0.33],
     [0.77, 0.25, 0.10],
     [0.05, 0.80, 0.55]]
)
inputs

tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.7700, 0.2500, 0.1000],
        [0.0500, 0.8000, 0.5500]])

In [59]:
query = inputs[1]
print("query:              ", query)
attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
  attn_scores_2[i] = torch.dot(x_i, query)
print("attention score 2:  ", attn_scores_2)
attn_weights_2 = attn_scores_2.softmax(dim=0)
print("attention weight 2: ", attn_weights_2)

query = inputs[1]
context_vec_2 = torch.zeros(query.shape)
for i, x_i in enumerate(inputs):
  print(attn_weights_2[i], x_i, attn_weights_2[i]*x_i)
  context_vec_2 += attn_weights_2[i]*x_i
print("context vec 2:      ", context_vec_2)

query:               tensor([0.5500, 0.8700, 0.6600])
attention score 2:   tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])
attention weight 2:  tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
tensor(0.1385) tensor([0.4300, 0.1500, 0.8900]) tensor([0.0596, 0.0208, 0.1233])
tensor(0.2379) tensor([0.5500, 0.8700, 0.6600]) tensor([0.1308, 0.2070, 0.1570])
tensor(0.2333) tensor([0.5700, 0.8500, 0.6400]) tensor([0.1330, 0.1983, 0.1493])
tensor(0.1240) tensor([0.2200, 0.5800, 0.3300]) tensor([0.0273, 0.0719, 0.0409])
tensor(0.1082) tensor([0.7700, 0.2500, 0.1000]) tensor([0.0833, 0.0270, 0.0108])
tensor(0.1581) tensor([0.0500, 0.8000, 0.5500]) tensor([0.0079, 0.1265, 0.0870])
context vec 2:       tensor([0.4419, 0.6515, 0.5683])


In [81]:
#  SIMPLIFIED
attn_scores  = inputs @ inputs.T
attn_weights = attn_scores.softmax(dim=-1)
contexts_vec = attn_weights @ inputs
contexts_vec

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])

In [132]:
import torch.nn as nn

x_2 = inputs[1]
d_in = inputs.shape[1]
d_out = 2

class SelfAttentionV1(nn.Module):
  def __init__(self, d_in, d_out, weights):
    super().__init__()
    self.W_query = nn.Parameter(weights[0])
    self.W_key   = nn.Parameter(weights[1])
    self.W_value = nn.Parameter(weights[2])

  def forward(self, x):
    keys    = x @ self.W_key
    queries = x @ self.W_query
    values  = x @ self.W_value
    attn_scores = queries @ keys.T
    attn_weights = torch.softmax(
        attn_scores / keys.shape[-1] ** 0.5, dim=1
    )
    context_vec = attn_weights @ values
    return context_vec

class SelfAttentionV2(nn.Module):
  def __init__(self, d_in, d_out, qkv_bias=False):
    super().__init__()
    self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

  def forward(self, x):
    keys    = self.W_key(x)
    queries = self.W_query(x)
    values  = self.W_value(x)
    attn_scores = queries @ keys.T
    attn_weights = torch.softmax(
        attn_scores / keys.shape[-1] ** 0.5, dim=1
    )
    context_vec = attn_weights @ values
    return context_vec

  def give_weights(self):
    return self.W_query.weight.T, self.W_key.weight.T, self.W_value.weight.T

torch.manual_seed(789)

sa_v2 = SelfAttentionV2(d_in, d_out)
print(sa_v2(inputs))
print()
sa_v1 = SelfAttentionV1(d_in, d_out, sa_v2.give_weights())
print(sa_v1(inputs))

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


In [125]:
sa_v2.give_weight()

Parameter containing:
tensor([[-0.4900, -0.3503, -0.2120],
        [-0.1135, -0.4404,  0.3780]], requires_grad=True)